# 🌀 07 · Tomography: tilt series, back-projection and the missing wedge
### How a stack of tilted 2D pictures becomes a 3D cell — and which information is lost for ever

**BiologyToResearch** · Part B — Imaging physics · Module 07 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/07_tomography_missing_wedge.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-image |
| **Prerequisites** | [03 · Fourier transform](03_fourier_frequency.ipynb) (frequencies, FFT, filters) and [06 · Image formation](06_electron_image_formation.ipynb) (projection approximation, dose). No calculus needed. |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Han *et al.* 2019 (joint marker-free tilt-series alignment, ISMB); Zeng *et al.* 2022 (2D annotated tilt series → 3D detection, J Comput Biol); optional: MicroFM 2026 (anisotropic resolution in light microscopy) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All experiments in this notebook are small, controlled *toy versions* of published methods. They illustrate the core
> idea of each paper; they do **not** reproduce the papers' numbers. **All molecules and images are generated by code**
> (nothing is downloaded): realistic stand-ins with the size and symmetry of the PDB entries they are named after.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why tomography tilts the specimen about one axis, and why tilting stops near ±60–70°.
2. **Verify** numerically the *projection-slice theorem*: each tilt image fills one line (2D) or plane (3D) of Fourier space.
3. **Implement** simple and weighted back-projection (WBP), and explain why the ramp filter $|k|$ is needed.
4. **Describe** the *missing wedge* $M(\boldsymbol\xi)=\mathbb 1\left(|\xi_z|\le|\xi_x|\tan\theta_{\max}\right)$ and **measure** the
   elongation it causes along the beam direction for different tilt ranges.
5. **Reason** about dose: why splitting a fixed dose over more tilts does not add noise, and why dose-symmetric schemes exist.
6. **Implement** a toy iterative reconstruction (SIRT) and a toy **tilt-series alignment** by projection matching, and
   explain how Han *et al.* (2019) *jointly* refine landmarks and projection parameters.
7. **Compare** segmenting in 3D with segmenting the 2D tilt images and mapping the masks back into 3D (Zeng *et al.* 2022).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, load tools and molecules | 5 min |
| 1 · Tilt geometry | thickness grows as 1/cos θ | 10 min |
| 2 · The projection-slice theorem | one projection = one line through Fourier space | 15 min |
| 3 · Back-projection | smear back, then sharpen with the ramp filter | 15 min |
| 4 · The missing wedge | see the empty wedge, vanishing membranes, a 3D reconstruction of zoo molecules | 15 min |
| 5 · Tilt range and elongation | measure how much ±40/60/90° matter | 10 min |
| 6 · Dose and tilt schemes | fixed-dose experiment, dose-symmetric acquisition | 10 min |
| 7 · Iterative reconstruction (SIRT) | forward-project, compare, correct | 10 min |
| 8 · Tilt-series alignment | projection matching + landmarks, Han *et al.* 2019 | 20 min |
| 9 · 2D labels → 3D | segment tilt images instead of the tomogram, Zeng *et al.* 2022 | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A single-particle cryo-EM study can afford to photograph *millions of identical, purified* molecules, each once. A cell
cannot be purified: every bacterium, every synapse, every mitochondrion is **unique**. To see it in 3D we must photograph
**the same frozen cell from many directions**, by tilting the microscope stage step by step — typically from −60° to
+60° in 2–3° steps. A computer then combines this **tilt series** into a 3D image, the **tomogram**.

Two problems make this harder than it sounds. First, the stage cannot tilt all the way to ±90°: the specimen holder and
the ever-thicker slab of ice block the beam. Directions that were never photographed leave a **missing wedge** of
information — membranes facing the "wrong" way literally disappear from the tomogram. Second, the stage wobbles: each
tilt image is shifted by a few nanometres, and these shifts must be found (**tilt-series alignment**) before any
reconstruction is sharp. This notebook builds both problems — and their solutions — from scratch.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell checks what computer you are on.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "(not needed here)")
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("skimage", "scikit-image")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from skimage.filters import threshold_otsu

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py), so that the
notebook runs on its own. Two of them are the stars of today: `project_tilt_series` (the microscope: rotate the volume
about the y axis and sum along the beam, z) and `weighted_back_projection` (the reconstruction). You will first rebuild
both yourself, in 2D, to see exactly what they do.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Build and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def project_tilt_series(vol: np.ndarray, angles_deg) -> np.ndarray:
    """Tilt-series projections of vol[z, y, x] about the y axis → array[n_tilts, y, x]."""
    return np.stack([ndimage.rotate(vol, a, axes=(0, 2), reshape=False, order=1).sum(axis=0)
                     for a in angles_deg]).astype(np.float32)


def weighted_back_projection(tilts: np.ndarray, angles_deg, nz: int) -> np.ndarray:
    """Weighted (ramp-filtered) back-projection of a y-axis tilt series → vol[nz, y, x]."""
    n, ny, nx = tilts.shape
    ramp = np.abs(np.fft.fftfreq(nx))[None, None, :]
    filt = np.real(np.fft.ifft(np.fft.fft(tilts, axis=2) * ramp, axis=2))
    rec = np.zeros((nz, ny, nx), np.float32)
    for img, a in zip(filt, angles_deg):
        smear = np.broadcast_to(img[None], (nz, ny, nx))
        rec += ndimage.rotate(smear, -a, axes=(0, 2), reshape=False, order=1)
    return rec / n
print("🧰 toolkit loaded")

### Data: the molecular zoo

We use the course's four "star" molecules (code-generated stand-ins for the PDB entries, 10 Å = 1 nm per voxel),
blurred to ~1 nm and boxed in 32 × 32 × 32 voxels.

In [ ]:
# ── Load the cast of molecules ──
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]          # ribosome, 20S proteasome, capped proteasome, RuBisCO
zoo = load_zoo(CAST, size=32, sigma=1.0)
print("box: 32³ voxels = 32 nm on a side (1 voxel = 1 nm)")

---
> 📍 **Where we are.** In [Module 03](03_fourier_frequency.ipynb) you learned that an image can be written as a sum of
> waves (its Fourier transform), and in [Module 06](06_electron_image_formation.ipynb) that an electron micrograph is,
> to a good approximation, a **projection** of the specimen's density along the beam, blurred by the CTF and buried in
> noise. This module asks: if we record projections of the same specimen at many tilt angles, how do we get the 3D
> density back, what is lost, and how do we fix misaligned images? Everything later in the course — subtomogram
> alignment (11), averaging (13), deep learning (15–24) — works on the tomograms built here, *with their missing wedge*.

---
## 1 · Tilt geometry: why the stage stops near ±60°

🧫 A cryo-ET specimen is a thin **slab** of vitreous ice: a whole small bacterium (≈ 200–500 nm thick) or a
**lamella** milled from a bigger cell with a focused ion beam (≈ 100–250 nm). The beam travels along **z**; we tilt the
slab about the **y axis** (the *tilt axis*, lying in the slab).

When the slab of thickness $t$ is tilted by angle $\theta$, the electrons cross a longer path:

$$t(\theta)=\frac{t}{\cos\theta}.$$

*In words:* at 60° the beam crosses **twice** the ice it crossed at 0°; at 70°, almost three times. Thicker ice means
more electrons scatter *inelastically* (they lose energy and blur the image), so fewer useful electrons reach the
detector. A common rule of thumb is that the fraction of electrons passing without inelastic scattering is about
$e^{-t(\theta)/\Lambda}$, with an inelastic mean free path $\Lambda$ of roughly 300–400 nm at 300 kV (we use 350 nm).

In [ ]:
# ── 1.1 Path length and useful electrons versus tilt ──
theta = np.linspace(0, 80, 161)
LAMBDA_NM = 350.0                                  # inelastic mean free path at 300 kV (approximate)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
# (a) schematic of a tilted slab
for ang, col in [(0, "tab:blue"), (30, "tab:orange"), (60, "tab:red")]:
    a = np.radians(ang)
    corners = np.array([[-1.2, -0.12], [1.2, -0.12], [1.2, 0.12], [-1.2, 0.12], [-1.2, -0.12]])
    rot = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    xy = corners @ rot.T
    axes[0].plot(xy[:, 0], xy[:, 1], color=col, label=f"tilt {ang}°")
axes[0].annotate("", xy=(0, -0.9), xytext=(0, 0.9), arrowprops=dict(arrowstyle="->", lw=2, color="k"))
axes[0].text(0.08, 0.75, "beam (z)", fontsize=8)
axes[0].set_aspect("equal"); axes[0].axis("off"); axes[0].legend(fontsize=7, loc="lower right")
axes[0].set_title("a slab tilted about y (into the page)")
# (b) relative path length
axes[1].plot(theta, 1 / np.cos(np.radians(theta)), color="tab:purple")
for a, ytxt in ((60, 4.0), (70, 5.2)):
    axes[1].axvline(a, color="gray", ls=":", lw=1)
    axes[1].text(a - 1.5, ytxt, f"{a}°: ×{1 / np.cos(np.radians(a)):.1f}", fontsize=7, ha="right")
axes[1].set_ylim(1, 6); axes[1].set_xlabel("tilt angle θ (°)"); axes[1].set_ylabel("path length / thickness")
axes[1].set_title("path length = t / cos θ")
# (c) fraction of unscattered (useful) electrons
for t_nm in (100, 200, 300):
    axes[2].plot(theta, np.exp(-t_nm / (LAMBDA_NM * np.cos(np.radians(theta)))), label=f"t = {t_nm} nm")
axes[2].set_xlabel("tilt angle θ (°)"); axes[2].set_ylabel("fraction without inelastic scattering")
axes[2].set_title(f"useful electrons (Λ ≈ {LAMBDA_NM:.0f} nm)"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()
for t_nm in (100, 200, 300):
    f0, f60 = (np.exp(-t_nm / (LAMBDA_NM * np.cos(np.radians(a)))) for a in (0, 60))
    print(f"t = {t_nm} nm: useful fraction {f0:.2f} at 0°  →  {f60:.2f} at 60°")

🔍 **What just happened?** Tilting is "free" near 0° but expensive at high tilt: the path length explodes as
$1/\cos\theta$, so the high-tilt images are darker and noisier — and above ~60–70° the holder and grid bars block the
beam anyway. **The views beyond the maximum tilt are simply never recorded.** Keep this in mind: in Section 4 these
missing views become the *missing wedge*.

💡 That is also why cells are thinned into **lamellae** before tomography: a 150 nm lamella keeps far more useful
electrons at 60° than a 300 nm whole cell.

---
## 2 · The projection-slice theorem: one tilt image = one line through Fourier space

To keep pictures simple we first work in **2D**: think of one **xz slice** of the specimen (perpendicular to the tilt
axis). Each tilt image then contributes a 1D **profile** to this slice. Tilting the specimen by $\theta$ and summing along
the beam gives the projection

$$p_\theta(u)=\iint f(x,z)\;\delta\!\left(u-x\cos\theta-z\sin\theta\right)\,dx\,dz ,$$

*in words:* the detector pixel $u$ collects all density lying on the straight line that the beam takes through the
tilted specimen (the $\delta$ just means "only points on that line count").

The **projection-slice theorem** (also called the *Fourier slice* or *central section* theorem) says:

$$\hat p_\theta(k)\;=\;\hat f\big(k_x=k\cos\theta,\;k_z=k\sin\theta\big).$$

*In words:* the 1D Fourier transform of a projection taken at angle $\theta$ equals the 2D Fourier transform of the
specimen **along a line through the origin at the same angle** (in 3D: a projection image gives a *plane* through the
origin). Each tilt image therefore fills one line of Fourier space; a tilt series fills a fan of lines.

Let us build a 2D test specimen from zoo molecules (each one flattened along y) plus a **membrane vesicle** (a ring),
and check the theorem numerically.

In [ ]:
# ── 2.1 Helpers for 2D tomography (the 1D-projection version of the toolkit functions) ──
def project_2d(img: np.ndarray, angles_deg) -> np.ndarray:
    """Projections of a 2D slice img[z, x]: rotate by each angle in the (z, x) plane, sum along z → sinogram[n, x]."""
    return np.stack([ndimage.rotate(img, a, axes=(0, 1), reshape=False, order=1).sum(axis=0)
                     for a in angles_deg])


def ramp_filter(sino: np.ndarray) -> np.ndarray:
    """Multiply each projection's 1D Fourier transform by |k| (the 'ramp'); returns real filtered projections."""
    ramp = np.abs(np.fft.fftfreq(sino.shape[-1]))
    return np.real(np.fft.ifft(np.fft.fft(sino, axis=-1) * ramp, axis=-1))


def back_project_2d(sino: np.ndarray, angles_deg, nz: int) -> np.ndarray:
    """Smear every 1D profile back along its beam direction and add them up (no filtering) → img[nz, nx]."""
    nx = sino.shape[1]
    rec = np.zeros((nz, nx))
    for p, a in zip(sino, angles_deg):
        rec += ndimage.rotate(np.broadcast_to(p[None], (nz, nx)), -a, axes=(0, 1), reshape=False, order=1)
    return rec / len(angles_deg)


def wbp_2d(sino: np.ndarray, angles_deg, nz: int) -> np.ndarray:
    """Weighted back-projection in 2D = ramp filter, then back-project."""
    return back_project_2d(ramp_filter(sino), angles_deg, nz)


def paste_2d(img: np.ndarray, patch: np.ndarray, cz: int, cx: int) -> None:
    """Add ``patch`` into ``img`` (in place), centred at row cz, column cx."""
    hz, hx = np.array(patch.shape) // 2
    img[cz - hz:cz - hz + patch.shape[0], cx - hx:cx - hx + patch.shape[1]] += patch

In [ ]:
# ── 2.2 A 2D test specimen: four molecules and a vesicle (96 × 96 nm, 1 nm pixels) ──
N2 = 96
phantom = np.zeros((N2, N2))
for pid, (cz, cx) in {"4v4r": (50, 24), "3j9i": (70, 60), "1bxn": (24, 42), "6utj": (72, 32)}.items():
    paste_2d(phantom, zoo[pid].sum(axis=1) / zoo[pid].sum(axis=1).max(), cz, cx)  # flatten along y
zz, xx = np.mgrid[0:N2, 0:N2]
ring = np.exp(-((np.hypot(zz - 34, xx - 66) - 13) ** 2) / (2 * 1.3 ** 2))    # a vesicle membrane (~4 nm thick)
phantom += 0.8 * ring
phantom /= phantom.max()
inside = np.hypot(zz - (N2 - 1) / 2, xx - (N2 - 1) / 2) < N2 / 2 - 1
assert phantom[~inside].max() < 0.05, "keep the specimen inside the inscribed circle (it must not rotate out of the box)"

fig, ax = plt.subplots(figsize=(3.6, 3.4))
ax.imshow(phantom)
for lab, (tz, tx) in {"ribosome": (33, 20), "20S": (84, 64), "RuBisCO": (13, 42), "capped 20S": (93, 32),
                      "vesicle": (35, 66)}.items():                   # label positions (z, x) in nm
    ax.text(tx, tz, lab, color="tab:orange", fontsize=7, ha="center", va="center")
ax.set_title("2D test specimen f(x, z)"); ax.set_xlabel("x (nm)"); ax.set_ylabel("z (nm)  — beam direction")
plt.show()

In [ ]:
# ── 2.3 Verify the projection-slice theorem ──
P0 = project_2d(phantom, [0])[0]
F2 = np.fft.fft2(phantom)
err0 = np.abs(np.fft.fft(P0) - F2[0]).max() / np.abs(F2).max()
print(f"θ = 0°: FFT(projection) vs row k_z = 0 of the 2D FFT → max relative difference {err0:.1e}  (exact)")

PAD = 4                                            # zero-padding makes Fourier space finely sampled for interpolation
padded = np.zeros((N2 * PAD, N2 * PAD)); padded[:N2, :N2] = phantom
F2p = np.fft.fftshift(np.fft.fft2(padded))
c = N2 * PAD // 2
k = np.arange(N2) - N2 // 2                        # frequencies of the 96-sample projection, in units of 1/96
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.1))
for ax, deg in zip(axes, [30, -45, 60]):
    th = np.radians(deg)
    proj_ft = np.fft.fftshift(np.fft.fft(project_2d(phantom, [deg])[0]))
    line = (ndimage.map_coordinates(F2p.real, [k * PAD * np.sin(th) + c, k * PAD * np.cos(th) + c], order=3)
            + 1j * ndimage.map_coordinates(F2p.imag, [k * PAD * np.sin(th) + c, k * PAD * np.cos(th) + c], order=3))
    rel = np.linalg.norm(np.abs(proj_ft) - np.abs(line)) / np.linalg.norm(np.abs(proj_ft))
    ax.semilogy(k / N2, np.abs(proj_ft), lw=2, label="1D FFT of projection")
    ax.semilogy(k / N2, np.abs(line), "--", lw=1.2, label="2D FFT sampled on the line")
    ax.set_title(f"θ = {deg}°  (difference {100 * rel:.1f} %)"); ax.set_xlabel("k (cycles/nm)")
    print(f"θ = {deg:4d}°: relative difference of magnitudes = {100 * rel:.2f} %")
    assert rel < 0.05, "the projection-slice theorem should hold to within interpolation error"
axes[0].set_ylabel("|Fourier amplitude|"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** At 0° the match is exact (the projection's spectrum *is* the $k_z=0$ row of the 2D
spectrum). At other angles the two curves overlap to within a fraction of a percent; the tiny difference is
interpolation error (pixels are square, lines are slanted). So **each tilt image measures one central line** of the
specimen's Fourier transform. A tilt series from −60° to +60° measures a *fan* of lines — let us draw it.

In [ ]:
# ── 2.4 Which part of Fourier space does a tilt series fill? ──
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8))
for a in np.arange(-60, 61, 3):
    t = np.radians(a)
    axes[0].plot([-0.5 * np.cos(t), 0.5 * np.cos(t)], [-0.5 * np.sin(t), 0.5 * np.sin(t)], color="tab:blue", lw=0.7)
edge = 0.5 / np.tan(np.radians(60))                    # where the wedge boundary meets k_z = ±0.5
for sgn in (1, -1):
    axes[0].fill([0, -edge, edge], [0, sgn * 0.5, sgn * 0.5], color="tab:red", alpha=0.2)
axes[0].text(0, 0.36, "missing\nwedge", color="tab:red", ha="center", fontsize=8)
axes[0].set_xlim(-0.5, 0.5); axes[0].set_ylim(-0.5, 0.5); axes[0].set_aspect("equal")
axes[0].set_xlabel("$k_x$ (cycles/nm)"); axes[0].set_ylabel("$k_z$ (cycles/nm)")
axes[0].set_title("measured lines: tilts −60°…+60° every 3°")
mask2d = missing_wedge_mask((N2, N2), 60)
axes[1].imshow(np.fft.fftshift(mask2d), origin="lower", extent=(-0.5, 0.5, -0.5, 0.5), cmap="viridis")
axes[1].set_xlabel("$k_x$ (cycles/nm)"); axes[1].set_ylabel("$k_z$ (cycles/nm)")
axes[1].set_title("toolkit missing_wedge_mask (yellow = measured)")
plt.tight_layout(); plt.show()
print(f"fraction of the 2D Fourier plane measured with ±60°: {mask2d.mean():.2f}  "
      f"(within a circle it would be 120/180 = {120 / 180:.2f}; the square grid's corners add a little)")

🔍 **What just happened?** The blue lines (one per tilt image) cover Fourier space except a double **wedge** around the
$k_z$ axis — the directions that would need tilts beyond ±60°. The toolkit's `missing_wedge_mask` encodes exactly this
region with the rule

$$M(\boldsymbol\xi)=\mathbb 1\big(|\xi_z|\le|\xi_x|\tan\theta_{\max}\big),$$

*in words:* a Fourier coefficient is measured if its direction is at most $\theta_{\max}$ away from the $x$ axis
($\mathbb 1(\cdot)$ is 1 when the condition holds and 0 otherwise). With $\theta_{\max}=60°$, one third of all directions
are never measured. Notice also that near the origin the lines crowd together, while far from the origin they spread
apart — remember this for the next section.

---
## 3 · Back-projection: smear back, then sharpen

How do we invert projections? The most intuitive idea is **back-projection**: take each 1D profile, *smear* it back
along the direction the beam came from, and add up all the smears. Where the smears from many angles agree, density
builds up.

Simple back-projection is blurry, and Fourier space tells us why: the measured lines are dense near the origin (low
frequencies) and sparse far away (high frequencies). Two neighbouring lines separated by an angle $\Delta\theta$ are a
distance $|k|\,\Delta\theta$ apart at frequency $|k|$, so the sampling density falls like $1/|k|$ — low frequencies are
over-counted. **Weighted back-projection (WBP)** compensates by multiplying each projection's spectrum by the **ramp**
$|k|$ before smearing:

$$f_{\text{WBP}}(x,z)=\sum_{j}\Big[\mathcal F^{-1}\big(|k|\,\hat p_{\theta_j}(k)\big)\Big]\big(x\cos\theta_j+z\sin\theta_j\big).$$

*In words:* sharpen every tilt image with the ramp filter, then smear it back along its beam direction and sum. This is
exactly what the toolkit's `weighted_back_projection` does in 3D.

In [ ]:
# ── 3.1 Simple vs weighted back-projection with a COMPLETE angular range (−90°…+88°) ──
ang_full = np.arange(-90, 90, 2.0)
sino_full = project_2d(phantom, ang_full)
bp = back_project_2d(sino_full, ang_full, N2)
wbp_full = wbp_2d(sino_full, ang_full, N2)
print(f"correlation with the true specimen:  simple BP {pearson(bp, phantom):.3f}   |   WBP {pearson(wbp_full, phantom):.3f}")

fig, axes = plt.subplots(1, 4, figsize=(11, 3.0))
axes[0].imshow(sino_full, aspect="auto", extent=(0, N2, ang_full[-1], ang_full[0]))
axes[0].set_title("sinogram (all tilt profiles)"); axes[0].set_xlabel("detector u (nm)"); axes[0].set_ylabel("tilt θ (°)")
kk = np.fft.fftshift(np.fft.fftfreq(N2))
axes[1].plot(kk, np.abs(kk), color="tab:purple"); axes[1].set_title("ramp filter |k|")
axes[1].set_xlabel("k (cycles/nm)"); axes[1].set_ylabel("weight")
axes[2].imshow(bp); axes[2].set_title("simple back-projection")
axes[3].imshow(wbp_full); axes[3].set_title("weighted back-projection")
for ax in axes[2:]:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("z (nm)")
plt.tight_layout(); plt.show()
assert pearson(wbp_full, phantom) > 0.95, "with a full angular range WBP should be nearly perfect"

🔍 **What just happened?** The **sinogram** (left) stacks all profiles: each point of the specimen traces a sine wave
(hence the name), because its projected position is $x\cos\theta+z\sin\theta$. Simple back-projection gives a halo-blurred
picture (correlation ≈ 0.6); the ramp filter turns it into an almost perfect reconstruction. **With a complete set of
angles, tomography works.** The trouble starts when angles are missing.

✏️ **Your turn.** Change the step in `np.arange(-90, 90, 2.0)` to `10.0` (only 18 views) and re-run 3.1. What do the
streaks look like? (This is *angular undersampling*; Section 6 quantifies it.)

---
## 4 · The missing wedge

### 4.1 In 2D: elongation and vanishing membranes

Now the realistic case: tilts only from −60° to +60° in 2° steps (61 images). We compare the WBP reconstruction with a
shortcut used throughout the course — take the true specimen and simply **zero the Fourier coefficients inside the
wedge** with `apply_fourier_mask(f, missing_wedge_mask(...))`. If the projection-slice theorem is right, the two should
look the same.

In [ ]:
# ── 4.1 WBP with ±60° versus the Fourier-mask shortcut ──
ang60 = np.arange(-60, 61, 2.0)
wbp60 = wbp_2d(project_2d(phantom, ang60), ang60, N2)
masked = apply_fourier_mask(phantom, missing_wedge_mask(phantom.shape, 60))
print(f"corr(WBP ±60°, truth)          = {pearson(wbp60, phantom):.3f}")
print(f"corr(Fourier-masked truth, truth) = {pearson(masked, phantom):.3f}")
print(f"corr(WBP ±60°, Fourier-masked)  = {pearson(wbp60, masked):.3f}   ← the shortcut is an excellent model of WBP")

fig, axes = plt.subplots(1, 4, figsize=(11, 3.0))
for ax, im, t in zip(axes, [phantom, wbp60, masked], ["truth", "WBP, tilts ±60°", "truth × wedge mask (shortcut)"]):
    ax.imshow(im); ax.set_title(t); ax.set_xlabel("x (nm)"); ax.set_ylabel("z (nm)")
spec = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(wbp60))))
axes[3].imshow(spec, origin="lower", extent=(-0.5, 0.5, -0.5, 0.5), cmap="viridis",
               vmin=np.percentile(spec, 5), vmax=np.percentile(spec, 99.5))
axes[3].set_title("log |FFT| of the WBP"); axes[3].set_xlabel("$k_x$"); axes[3].set_ylabel("$k_z$")
plt.tight_layout(); plt.show()
assert pearson(wbp60, masked) > 0.95

🔍 **What just happened?** Three things to notice — they are the daily bread of cryo-ET:

1. **The Fourier spectrum of the reconstruction is empty in a wedge** around the $k_z$ axis (right panel, dark
   "bow-tie"). Nothing in the reconstruction algorithm can bring these coefficients back — they were never measured.
2. **Objects are elongated along z** (the beam direction) and show X-shaped streaks: compact molecules look stretched
   vertically.
3. **The vesicle has lost its top and bottom.** A membrane patch that is *horizontal* (thin along z) has its Fourier
   content along the $k_z$ axis — right inside the missing wedge — so it disappears; vertical patches survive. Real
   tomograms show exactly this: vesicles look like "( )" rather than "O".

🧫 This is why biologists must be careful when interpreting membranes, filaments and molecular shapes in tomograms:
what you see depends on how the object is oriented relative to the tilt axis and beam.

### 4.2 In 3D: a tomogram of zoo molecules

In 3D the measured set is a fan of **planes** (each containing the $k_y$ axis, the tilt axis). Let us place a ribosome,
a 20S proteasome and two RuBisCO molecules in random orientations in a 64 nm cube, record a ±60° tilt series with the
toolkit's `project_tilt_series`, and reconstruct with `weighted_back_projection`.

In [ ]:
# ── 4.2 Build a small 3D specimen and reconstruct it ──
def place_template(vol: np.ndarray, template: np.ndarray, center_zyx, R: np.ndarray) -> None:
    """Add a rotated copy of ``template`` into ``vol`` (in place), its box centred at ``center_zyx`` (clipped at edges)."""
    sub = rotate_volume(template, R)
    lo = np.array(center_zyx) - np.array(sub.shape) // 2
    sl_v, sl_s = [], []
    for a in range(3):
        a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], vol.shape[a])
        sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
    vol[tuple(sl_v)] += sub[tuple(sl_s)]


S3 = 64
specimen = np.zeros((S3, S3, S3), np.float32)
layout = [("4v4r", (34, 32, 20)), ("3j9i", (30, 32, 46)), ("1bxn", (12, 32, 38)), ("1bxn", (52, 32, 40))]
rng3 = np.random.default_rng(SEED + 7)
for pid, cen in layout:
    place_template(specimen, zoo[pid], cen, random_rotation(rng3))
t0 = time.time()
tilts60 = project_tilt_series(specimen, ang60)                     # the microscope: 61 projections [n, y, x]
rec60 = weighted_back_projection(tilts60, ang60, S3)               # the reconstruction [z, y, x]
print(f"tilt series {tilts60.shape} and WBP {rec60.shape} in {time.time() - t0:.1f} s")
print(f"corr(WBP ±60°, truth) = {pearson(rec60, specimen):.3f};  "
      f"corr(WBP, Fourier-masked truth) = {pearson(rec60, apply_fourier_mask(specimen, missing_wedge_mask(specimen.shape, 60))):.3f}")

In [ ]:
# ── 4.3 Look at the 3D reconstruction: xy is fine, xz is stretched ──
y0 = 32
fig, axes = plt.subplots(2, 3, figsize=(9.6, 6.2))
for row, (vol, lab) in enumerate([(specimen, "truth"), (rec60, "WBP ±60°")]):
    axes[row, 0].imshow(vol[34]); axes[row, 0].set_title(f"{lab}: xy slice (z = 34 nm)")
    axes[row, 1].imshow(vol[:, y0, :]); axes[row, 1].set_title(f"{lab}: xz slice (y = {y0} nm)")
    axes[row, 0].set_xlabel("x (nm)"); axes[row, 0].set_ylabel("y (nm)")
    axes[row, 1].set_xlabel("x (nm)"); axes[row, 1].set_ylabel("z (nm)")
axes[0, 2].imshow(tilts60[30]); axes[0, 2].set_title("tilt image at 0°"); axes[0, 2].set_xlabel("x"); axes[0, 2].set_ylabel("y")
spec3 = np.log1p(np.abs(np.fft.fftshift(np.fft.fftn(rec60)))).mean(axis=1)      # (k_z, k_x), averaged over k_y
axes[1, 2].imshow(spec3, origin="lower", extent=(-0.5, 0.5, -0.5, 0.5), cmap="viridis",
                  vmin=np.percentile(spec3, 5), vmax=np.percentile(spec3, 99.5))
xs = np.linspace(-0.5, 0.5, 2)
for sgn in (1, -1):
    axes[1, 2].plot(xs, sgn * xs * np.tan(np.radians(60)), "w--", lw=1)
axes[1, 2].set_xlim(-0.5, 0.5); axes[1, 2].set_ylim(-0.5, 0.5)
axes[1, 2].set_title("log |FFT| (mean over $k_y$)"); axes[1, 2].set_xlabel("$k_x$"); axes[1, 2].set_ylabel("$k_z$")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the **xy** slice (looking down the beam) the reconstruction resembles the truth; in the
**xz** slice every molecule is stretched along z and crossed by streaks. The 3D spectrum is empty between the white
dashed lines, $|k_z|>|k_x|\tan 60°$: the missing wedge (strictly, in 3D it is a wedge *for every* $k_y$ — a
"missing wedge" extruded along the tilt axis). The correlation with the Fourier-masked truth is much higher than with
the truth itself: the wedge, not the algorithm, is the main loss.

💡 **Why this matters for the rest of the course.** Every subtomogram you will align (Modules 12–13, 21) or classify
(13–17, 20) carries this wedge, and *different* copies of a molecule have their wedge in different orientations
relative to the molecule. Constrained correlation (Module 12) and wedge-aware averaging (Module 14) exist because of this
picture.

---
## 5 · How the tilt range controls elongation

How bad is the stretch? A classic way to answer is to reconstruct a **point** (a tiny bead) and measure the width of its
image — the *point-spread function* (PSF) — along z and along x. A widely used estimate of the elongation factor for a
single-axis tilt range $\pm\alpha$ (Radermacher 1988) is

$$e_{xz}=\sqrt{\frac{\alpha+\sin\alpha\cos\alpha}{\alpha-\sin\alpha\cos\alpha}}\qquad(\alpha\ \text{in radians}).$$

*In words:* the smaller the maximal tilt $\alpha$, the more the point is smeared along the beam relative to across it;
at $\alpha=90°$ there is no elongation ($e=1$).

In [ ]:
# ── 5.1 Measure the PSF elongation for several tilt ranges ──
def fwhm(profile: np.ndarray) -> float:
    """Full width at half maximum of a single-peaked 1D profile (linear interpolation between samples)."""
    p = profile / profile.max()
    i = int(np.argmax(p))
    lft = i
    while lft > 0 and p[lft] > 0.5:
        lft -= 1
    rgt = i
    while rgt < len(p) - 1 and p[rgt] > 0.5:
        rgt += 1
    x_l = lft + (0.5 - p[lft]) / (p[lft + 1] - p[lft])
    x_r = rgt - 1 + (p[rgt - 1] - 0.5) / (p[rgt - 1] - p[rgt])
    return x_r - x_l


def radermacher_elongation(tilt_max_deg: float) -> float:
    """Elongation factor e_xz of a single-axis tilt range ±tilt_max (Radermacher 1988)."""
    a = np.radians(tilt_max_deg)
    return float(np.sqrt((a + np.sin(a) * np.cos(a)) / (a - np.sin(a) * np.cos(a))))


point = np.zeros((128, 128)); point[64, 64] = 1.0
point = ndimage.gaussian_filter(point, 1.5)                       # a 'gold bead' of ~3.5 nm
ranges = [40, 50, 60, 70, 80, 90]
psfs, measured = {}, []
for tm in ranges:
    a = np.arange(-tm, tm + 0.5, 1.0) if tm < 90 else np.arange(-90, 90, 1.0)
    psfs[tm] = wbp_2d(project_2d(point, a), a, 128)
    measured.append(fwhm(psfs[tm][:, 64]) / fwhm(psfs[tm][64, :]))
    print(f"±{tm}°: measured FWHM_z / FWHM_x = {measured[-1]:.2f}   Radermacher e_xz = {radermacher_elongation(tm):.2f}")

fig, axes = plt.subplots(1, 4, figsize=(11, 2.9))
for ax, tm in zip(axes[:3], [40, 60, 90]):
    ax.imshow(psfs[tm][44:84, 44:84], cmap="gray"); ax.set_title(f"PSF, tilts ±{tm}°")
    ax.set_xlabel("x (nm)"); ax.set_ylabel("z (nm)")
fine = np.linspace(35, 90, 100)
axes[3].plot(fine, [radermacher_elongation(t) for t in fine], color="tab:purple", label="Radermacher formula")
axes[3].plot(ranges, measured, "o", color="tab:orange", label="measured (FWHM ratio)")
axes[3].set_xlabel("max tilt α (°)"); axes[3].set_ylabel("elongation z / x"); axes[3].legend(fontsize=7)
axes[3].set_title("elongation vs tilt range")
plt.tight_layout(); plt.show()
assert abs(measured[ranges.index(60)] - radermacher_elongation(60)) < 0.3

🔍 **What just happened?** A round bead becomes an X-shaped, vertically stretched blob; at ±60° it is about 1.5–1.7×
longer along z than along x, and our measurement follows the classic formula to within about 10 % (the exact number
depends on how one defines "width"). Going from ±60° to ±70° helps noticeably; going below ±50° hurts a lot.

### 5.2 The tilt range in 3D

In [ ]:
# ── 5.2 Reconstruct the 3D specimen with ±40°, ±60° and the full range ──
recs = {}
for tm in (40, 60, 90):
    a = np.arange(-tm, tm + 0.5, 2.0) if tm < 90 else np.arange(-90, 90, 2.0)
    recs[tm] = weighted_back_projection(project_tilt_series(specimen, a), a, S3)
fig, axes = plt.subplots(1, 4, figsize=(11, 2.9))
axes[0].imshow(specimen[:, y0, :]); axes[0].set_title("truth (xz)")
for ax, tm in zip(axes[1:], (40, 60, 90)):
    ax.imshow(recs[tm][:, y0, :])
    ax.set_title(f"±{tm}°: corr {pearson(recs[tm], specimen):.2f}")
for ax in axes:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("z (nm)")
plt.tight_layout(); plt.show()
for tm in (40, 60, 90):
    print(f"±{tm}°: corr with truth = {pearson(recs[tm], specimen):.3f}")

🔍 **What just happened?** With ±40° the molecules dissolve into vertical streaks; with the (physically impossible) full
range they are recovered almost perfectly. Real data sit around ±60°, which is why so much of the lab's work —
constrained correlation (Module 12), missing-wedge-aware deep networks (Modules 21–23) — deals with the wedge rather than
hoping it away.

✏️ **Your turn.** A *dual-axis* tilt series records a second series after rotating the grid by 90° in its plane, so the
missing **wedge** shrinks to a missing **pyramid**. Using `missing_wedge_mask`, build the dual-axis mask as the union
(logical OR) of the usual mask and the same mask with the roles of x and y swapped (hint: transpose axes 1 and 2), and
print the fraction of Fourier space measured in each case.

In [ ]:
# ── ✏️ Your turn: single-axis vs dual-axis coverage ──
m_single = missing_wedge_mask((32, 32, 32), 60)
m_dual = np.maximum(m_single, np.transpose(m_single, (0, 2, 1)))     # second series with the tilt axis along x
print(f"fraction of 3D Fourier space measured: single-axis {m_single.mean():.2f}, dual-axis {m_dual.mean():.2f}")

### 📄 Side box — the optical cousin of the missing wedge (optional)

**Paper**: Xingzu Zhan, Runmin Jiang, Vatsal Gupta, Tanush Swaminathan, Yanwen Wang, Genpei Zhang, Haili Wang, Min Xu.
*MicroFM: Physics-guided Flow Matching for Isotropic Microscopy Reconstruction.* CVPR 2026
([OpenReview record](https://openreview.net/forum?id=0CSA9nQavi)).

> ⚠️ We could not access the full text of this paper while writing; the description below is limited to the title and
> abstract. Read the paper before relying on details.

- **Question it answers:** in 3D **fluorescence (light) microscopy** — not cryo-EM — resolution along the optical axis
  (z) is much worse than in-plane, because the point-spread function is elongated. Can a computer restore isotropic
  resolution?
- **What the paper does (abstract level):** trains a *flow-matching* generative model on pairs synthesised with
  **physical PSFs matched to each microscope** (rather than generic Gaussian blur), with a continuous implicit geometry
  prior for volumetric consistency; evaluated on four fluorescence microscopy systems.
- **Connection to this module:** an elongated PSF along z is exactly what you measured in 5.1 — in cryo-ET it comes from
  missing *views*, in light microscopy from the optics. Both are "anisotropic resolution" problems; restoring them needs
  a good model of *how* the information was lost.
- **What our toy leaves out:** everything about the method itself (no generative model here).

---
## 6 · Dose and tilt schemes

🧫 Biological specimens tolerate only a limited **total electron dose** (roughly 100–150 e⁻/Å² for a whole cellular
tilt series) before radiation damage destroys fine detail. So the question is not "how many images?" but "how should a
*fixed* dose be split?".

**Dose fractionation.** Splitting a fixed total dose over $N$ tilts makes every image $N$ times dimmer (noisier), but the
reconstruction adds $N$ images back up. For a linear reconstruction such as WBP the noise in the final tomogram is
(approximately) **independent of $N$** — the *dose-fractionation theorem* of Hegerl & Hoppe (1976). Meanwhile, more tilts
sample Fourier space more finely. The **Crowther criterion** says that $N$ equally spaced projections over 180° of an
object of diameter $D$ support a resolution of about

$$d\approx\frac{\pi D}{N},$$

*in words:* the larger the object (or the thicker the slab), the more views you need for the same detail. Let us test both
ideas with a fixed total dose.

In [ ]:
# ── 6.1 Fixed total dose, different numbers of tilts ──
def noisy_tilt_series_fixed_dose(img: np.ndarray, angles_deg, sigma_one_image: float, rng) -> np.ndarray:
    """Projections with shot-like noise for a FIXED total dose split equally over the tilts.

    sigma_one_image is the noise std if the whole dose went into one image; with N tilts each image gets 1/N of the
    dose, so (in units of the signal) its noise std grows by √N.
    """
    sino = project_2d(img, angles_deg)
    return sino + rng.normal(0, sigma_one_image * np.sqrt(len(angles_deg)), sino.shape)


masked60 = apply_fourier_mask(phantom, missing_wedge_mask(phantom.shape, 60))     # best possible with ±60°
SIGMA_ONE = 0.05 * project_2d(phantom, [0])[0].std()
n_list = [5, 9, 17, 31, 61, 121]
res_clean, res_noisy, examples = [], [], {}
for n in n_list:
    a = np.linspace(-60, 60, n)
    r_clean = wbp_2d(project_2d(phantom, a), a, N2)
    r_noisy = wbp_2d(noisy_tilt_series_fixed_dose(phantom, a, SIGMA_ONE, np.random.default_rng(1)), a, N2)
    res_clean.append(pearson(r_clean, masked60)); res_noisy.append(pearson(r_noisy, masked60))
    examples[n] = r_noisy
    print(f"N = {n:3d} tilts (step {120 / (n - 1):5.1f}°): corr no noise {res_clean[-1]:.3f} | fixed total dose {res_noisy[-1]:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.0))
axes[0].semilogx(n_list, res_clean, "o-", label="no noise (sampling only)")
axes[0].semilogx(n_list, res_noisy, "s-", label="fixed total dose")
axes[0].set_xlabel("number of tilts N (±60°)"); axes[0].set_ylabel("corr with wedge-limited truth")
axes[0].set_title("splitting a fixed dose"); axes[0].legend(fontsize=7)
for ax, n in zip(axes[1:], (5, 61)):
    ax.imshow(examples[n]); ax.set_title(f"N = {n}, same total dose"); ax.set_xlabel("x (nm)"); ax.set_ylabel("z (nm)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the total dose fixed, the noisy curve never *falls* as we add tilts — the noise in the
reconstruction stays about the same (dose fractionation) — while the sampling improves until ~30 tilts, after which our
~70 nm specimen is sampled finely enough at this resolution (Crowther). Below that, few tilts produce the streaks you see
with $N=5$. Real acquisitions use 2–3° steps (41–61 images) for exactly this reason.

### 6.2 In which order should we collect the tilts?

Radiation damage accumulates: the *last* images are recorded on an already-damaged specimen. Three common orders:

| Scheme | Order | Consequence |
|---|---|---|
| unidirectional | −60° → +60° | the +60° side is the most damaged |
| bidirectional | 0° → +60°, then −2° → −60° | the whole negative half is recorded on a pre-exposed specimen |
| **dose-symmetric** (Hagen, Wan & Briggs 2017) | 0°, then alternate sides in small groups: +3, +6, −3, −6, +9, +12, … | the low tilts — thinnest ice, most information — get the lowest prior dose |

In [ ]:
# ── 6.2 Accumulated dose before each tilt, for three acquisition orders ──
def acquisition_order(scheme: str, step: float = 3.0, tmax: float = 60.0, group: int = 2) -> list:
    """Return the tilt angles in the order they are collected."""
    pos = list(np.arange(step, tmax + 0.1, step)); neg = [-a for a in pos]
    pos = [float(a) for a in pos]; neg = [float(a) for a in neg]
    if scheme == "unidirectional":
        return sorted(neg) + [0.0] + pos
    if scheme == "bidirectional":
        return [0.0] + pos + neg
    order, i = [0.0], 0                                     # dose-symmetric, groups of `group` per side
    while i < len(pos):
        order += pos[i:i + group] + neg[i:i + group]
        i += group
    return order


fig, ax = plt.subplots(figsize=(6.4, 3.2))
for scheme, col in [("unidirectional", "tab:gray"), ("bidirectional", "tab:blue"), ("dose-symmetric", "tab:red")]:
    order = acquisition_order(scheme)
    prior = {a: k for k, a in enumerate(order)}            # number of images taken before this one (≈ prior dose)
    angs = sorted(prior)
    ax.plot(angs, [100 * prior[a] / len(order) for a in angs], "o-", ms=3, color=col, label=scheme)
ax.set_xlabel("tilt angle (°)"); ax.set_ylabel("dose received before this image (% of total)")
ax.set_title("which tilts are recorded on a fresh specimen?"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("dose-symmetric order (first 11):", acquisition_order("dose-symmetric")[:11])

🔍 **What just happened?** Only the dose-symmetric scheme keeps the prior dose low for *all* low-tilt images, which
carry the most high-resolution information (thinnest ice, least damage). It is the standard for subtomogram averaging
at high resolution — the data the lab's averaging methods (Modules 12–14) consume.

---
## 7 · Iterative reconstruction (SIRT) in one picture

WBP is fast but noisy (the ramp amplifies high-frequency noise) and streaky. **Iterative** methods treat reconstruction
as solving the equations "projections of my guess = measured projections". Write $A$ for the projector (our
`project_2d`), $A^{\mathsf T}$ for unfiltered back-projection, $b$ for the measured sinogram. The **SIRT** update
(Simultaneous Iterative Reconstruction Technique; Gilbert 1972) is

$$x^{(k+1)}=x^{(k)}+\lambda\,C\,A^{\mathsf T}R\,\big(b-A\,x^{(k)}\big),$$

*in words:* project the current guess, compare with the data (the residual $b-Ax$), smear the residual back and add a
fraction of it; $R$ and $C$ divide by the ray lengths and by the number of rays through each pixel so the step has the
right size. A simple extra rule — **density cannot be negative** — can be enforced after each step.

In [ ]:
# ── 7.1 SIRT versus WBP on noisy data (±60°, 41 tilts) ──
def sirt_2d(sino: np.ndarray, angles_deg, n_iter: int = 30, relax: float = 1.0, positivity: bool = False,
            truth: np.ndarray | None = None):
    """Simultaneous Iterative Reconstruction Technique in 2D. Returns (image, list of corr with truth per iteration)."""
    n = sino.shape[1]
    ray_len = np.maximum(project_2d(np.ones((n, n)), angles_deg), 1.0)            # R⁻¹: length of each ray
    hits = np.maximum(back_project_2d(np.ones_like(sino), angles_deg, n), 1e-3)  # C⁻¹: rays through each pixel
    x, history = np.zeros((n, n)), []
    for _ in range(n_iter):
        resid = (sino - project_2d(x, angles_deg)) / ray_len
        x = x + relax * back_project_2d(resid, angles_deg, n) / hits
        if positivity:
            x = np.clip(x, 0, None)
        if truth is not None:
            history.append(pearson(x, truth))
    return x, history


ang41 = np.arange(-60, 61, 3.0)
sino41 = project_2d(phantom, ang41)
sino41_noisy = sino41 + np.random.default_rng(2).normal(0, 0.3 * sino41.std(), sino41.shape)
t0 = time.time()
rec_wbp = wbp_2d(sino41_noisy, ang41, N2)
rec_sirt, h_sirt = sirt_2d(sino41_noisy, ang41, 30, truth=phantom)
rec_sirtp, h_sirtp = sirt_2d(sino41_noisy, ang41, 30, positivity=True, truth=phantom)
print(f"done in {time.time() - t0:.1f} s | corr with truth: WBP {pearson(rec_wbp, phantom):.3f}, "
      f"SIRT best {max(h_sirt):.3f} (iteration {int(np.argmax(h_sirt)) + 1}), SIRT+positivity {h_sirtp[-1]:.3f}")

fig, axes = plt.subplots(1, 4, figsize=(11.5, 3.0))
axes[0].plot(range(1, 31), h_sirt, label="SIRT"); axes[0].plot(range(1, 31), h_sirtp, label="SIRT + positivity")
axes[0].axhline(pearson(rec_wbp, phantom), color="gray", ls="--", label="WBP")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("corr with truth"); axes[0].legend(fontsize=7)
axes[0].set_title("convergence")
for ax, im, t in zip(axes[1:], [rec_wbp, rec_sirt, rec_sirtp], ["WBP", "SIRT (30 it.)", "SIRT + positivity"]):
    ax.imshow(im); ax.set_title(t); ax.set_xlabel("x (nm)"); ax.set_ylabel("z (nm)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** SIRT starts smooth and adds detail with each iteration; on noisy data its quality peaks and
then slowly declines as it starts fitting the noise (*semi-convergence* — the iteration number acts as a smoothing
knob). Adding the physical constraint "no negative density" helps a lot in this toy because our specimen sits on an
empty background. Note what SIRT does **not** do: it cannot measure the missing wedge; it can only make a plausible
guess consistent with the constraints. (Learned priors — Modules 21–24 — make much stronger guesses, which is both their
power and their danger.)

---
## 8 · Tilt-series alignment: finding where each image moved

🧫 Between two tilt images the stage moves mechanically; the specimen drifts and is deformed by the beam. In practice each
raw image is displaced by several to hundreds of nanometres, and the tilt axis and image rotation are known only
approximately. If we reconstruct without correcting these errors, all the smears in back-projection miss each other.

Two families of solutions:

| | **Fiducial-based** | **Marker-free** |
|---|---|---|
| Landmarks | gold beads (5–15 nm) added to the sample | natural features of the specimen (patches, keypoints) |
| Pros | strong, unambiguous signal | no beads (beads can be absent in FIB lamellae, or hide structures) |
| Cons | beads cause streaks; need several per field | features change appearance with tilt; harder to track |
| Examples | IMOD fiducial model | patch tracking, **projection matching**, Han *et al.* 2019 |

**The projection model.** Han *et al.* (2019) write the position $(u,v)$ in image $j$ of a 3D point $(X,Y,Z)$ as

$$\begin{pmatrix}u\\v\end{pmatrix}=s_j\,\mathbf R_{\gamma_j}\,\mathbf P\,\mathbf R_{\beta_j}\mathbf R_{\alpha_j}\begin{pmatrix}X\\Y\\Z\end{pmatrix}+\mathbf t_j,
\qquad \mathbf P=\begin{pmatrix}1&0&0\\0&1&0\end{pmatrix},$$

*in words:* tilt the point (angles $\alpha_j$, $\beta_j$), drop the z coordinate (projection $\mathbf P$), rotate in the
image plane ($\gamma_j$), scale ($s_j$) and shift ($\mathbf t_j$). Alignment = estimating these parameters for every image.

**Our toy keeps one parameter per image:** the shift $t_j$ along x, in our 2D slice:

$$u_{ij}=X_i\cos\beta_j+Z_i\sin\beta_j+t_j .$$

### 8.1 Simulate a misaligned tilt series

In [ ]:
# ── 8.1 A misaligned, noisy tilt series ──
def fourier_shift_1d(profiles: np.ndarray, shifts) -> np.ndarray:
    """Shift each 1D profile by a (sub-pixel) amount using the Fourier shift theorem (Module 03)."""
    k = np.fft.fftfreq(profiles.shape[-1])
    ph = np.exp(-2j * np.pi * k[None, :] * np.asarray(shifts, float)[:, None])
    return np.real(np.fft.ifft(np.fft.fft(profiles, axis=-1) * ph, axis=-1))


beta = np.radians(ang60)
n_t = len(ang60)
rng8 = np.random.default_rng(5)
t_true = rng8.normal(0, 6.0, n_t)                                  # true shifts: ~6 nm standard deviation
clean_sino = project_2d(phantom, ang60)
measured = fourier_shift_1d(clean_sino, t_true)
measured += rng8.normal(0, 0.15 * measured.std(), measured.shape)
rec_misaligned = wbp_2d(measured, ang60, N2)
rec_perfect = wbp_2d(fourier_shift_1d(measured, -t_true), ang60, N2)
print(f"true shifts: rms {np.sqrt(np.mean(t_true ** 2)):.1f} nm, range {t_true.min():.1f} … {t_true.max():.1f} nm")
print(f"corr with truth: misaligned {pearson(rec_misaligned, phantom):.3f} | perfectly aligned {pearson(rec_perfect, phantom):.3f}")

### 8.2 Projection matching: align each image to a re-projection of the current tomogram

The idea (the *intensity-based* or *intrinsic* refinement): (1) reconstruct a preliminary tomogram with the current shift
estimates; (2) **re-project** it at each tilt angle; (3) find the shift that best matches each measured image to its
re-projection by **cross-correlation** (Module 03); (4) repeat. Two practical details:

- **Leave one out:** when estimating image $j$'s shift we re-project a tomogram built *without* image $j$, so the image is
  not simply compared with itself.
- **The gauge freedom.** Moving the whole 3D object by $(a,b)$ in $(x,z)$ changes every projected position by
  $a\cos\beta_j+b\sin\beta_j$ — exactly like a shift pattern. So two combinations of shifts can **never** be determined
  from the data: they only re-centre the tomogram. We therefore measure errors *after* removing the best-fitting
  $a\cos\beta_j+b\sin\beta_j$ component. (Han *et al.* prove a closely related statement: re-centring the 3D origin
  changes the translations but keeps the projections consistent.)

In [ ]:
# ── 8.2 Helpers: cross-correlation shift, gauge-free error, one projection-matching step ──
G_GAUGE = np.stack([np.cos(beta), np.sin(beta)], axis=1)            # the two undeterminable shift patterns
P_GAUGE = G_GAUGE @ np.linalg.pinv(G_GAUGE)                          # projector onto them


def xcorr_shift_1d(a: np.ndarray, b: np.ndarray, lowpass: float = 0.25) -> float:
    """Sub-pixel shift s such that a(u) ≈ b(u − s), from the peak of their (low-passed) cross-correlation."""
    k = np.abs(np.fft.fftfreq(len(a)))
    cc = np.real(np.fft.ifft(np.fft.fft(a) * np.conj(np.fft.fft(b)) * np.exp(-(k / lowpass) ** 2)))
    i, L = int(np.argmax(cc)), len(cc)
    y0, y1, y2 = cc[(i - 1) % L], cc[i], cc[(i + 1) % L]
    s = i + 0.5 * (y0 - y2) / (y0 - 2 * y1 + y2 + 1e-12)           # parabola through the 3 top points
    return s - L if s > L / 2 else s


def gauge_free_error(est: np.ndarray, truth: np.ndarray) -> float:
    """RMS shift error (nm) after removing the undeterminable a·cosβ + b·sinβ component."""
    e = est - truth
    return float(np.sqrt(np.mean((e - P_GAUGE @ e) ** 2)))


def projection_matching_step(meas: np.ndarray, est: np.ndarray, angles_deg) -> np.ndarray:
    """One round: WBP with current shifts → leave-one-out re-projection → cross-correlate → new shift estimates."""
    n = meas.shape[1]
    filt = ramp_filter(fourier_shift_1d(meas, -est))
    smears = [ndimage.rotate(np.broadcast_to(p[None], (n, n)), -a, axes=(0, 1), reshape=False, order=1)
              for p, a in zip(filt, angles_deg)]
    total = np.sum(smears, axis=0)
    new = np.empty(len(angles_deg))
    for j, a in enumerate(angles_deg):
        reproj = ndimage.rotate(total - smears[j], a, axes=(0, 1), reshape=False, order=1).sum(axis=0)
        new[j] = xcorr_shift_1d(meas[j], reproj)
    return new - P_GAUGE @ new                                       # fix the gauge (choose a centring)

In [ ]:
# ── 8.3 Run projection matching from a zero initial guess ──
t0 = time.time()
est_pm, err_pm = np.zeros(n_t), [gauge_free_error(np.zeros(n_t), t_true)]
for it in range(8):
    est_pm = projection_matching_step(measured, est_pm, ang60)
    err_pm.append(gauge_free_error(est_pm, t_true))
print(f"8 rounds in {time.time() - t0:.1f} s; gauge-free shift error (nm):", " → ".join(f"{e:.2f}" for e in err_pm))


def reconstruct_with(est: np.ndarray) -> np.ndarray:
    """WBP after undoing the estimated shifts; the gauge part is set to the truth so images are comparable."""
    e = est - t_true
    return wbp_2d(fourier_shift_1d(measured, -(est - P_GAUGE @ e)), ang60, N2)


rec_pm = reconstruct_with(est_pm)
print(f"corr with truth after projection matching: {pearson(rec_pm, phantom):.3f} "
      f"(perfect alignment {pearson(rec_perfect, phantom):.3f})")
assert err_pm[-1] < 0.5 * err_pm[0], "projection matching should at least halve the shift error"

🔍 **What just happened?** Starting from a badly blurred tomogram, a few rounds of "reconstruct → re-project →
cross-correlate" recover the shifts to about a nanometre or better and the reconstruction becomes nearly as good as with
perfect alignment. If you look at the error after each round, it does not go to zero: the tomogram is built from the
very shifts we are estimating, so smooth error patterns that *almost* look like the gauge are only weakly constrained and
can drift. This is one motivation for combining intensities with **landmarks**.

### 8.4 Landmarks (track-based alignment) and their least-squares fit

Suppose a tracker has followed $K$ landmarks (gold beads, or natural keypoints such as Han *et al.*'s SIFT features)
through the series, measuring their positions $u_{ij}$ with some localisation error. The model $u_{ij}=X_i\cos\beta_j+
Z_i\sin\beta_j+t_j$ is **linear** in all unknowns ($X_i$, $Z_i$, $t_j$), so we can solve it by least squares — the
simplest form of **bundle adjustment** (fitting 3D points and camera parameters together). In our toy the landmark
positions are generated directly from the model plus noise (we skip the tracking step).

In [ ]:
# ── 8.4 Landmark least squares: unknowns X_i, Z_i (per landmark) and t_j (per tilt) ──
def landmark_design(n_landmarks: int, beta_rad: np.ndarray) -> np.ndarray:
    """Design matrix of u_ij = X_i cosβ_j + Z_i sinβ_j + t_j (rows ordered landmark-major)."""
    n = len(beta_rad)
    A = np.zeros((n_landmarks * n, 2 * n_landmarks + n))
    for i in range(n_landmarks):
        rows = slice(i * n, (i + 1) * n)
        A[rows, 2 * i] = np.cos(beta_rad)
        A[rows, 2 * i + 1] = np.sin(beta_rad)
        A[rows, 2 * n_landmarks:] = np.eye(n)
    return A


def fit_shifts(u_meas: np.ndarray, beta_rad: np.ndarray, t_obs: np.ndarray | None = None, weight: float = 0.0):
    """Least-squares shifts from landmark tracks; optionally also from direct shift observations t_obs (weighted)."""
    K, n = u_meas.shape
    A, b = landmark_design(K, beta_rad), u_meas.ravel()
    if t_obs is not None:
        B = np.zeros((n, A.shape[1])); B[:, 2 * K:] = weight * np.eye(n)
        A, b = np.vstack([A, B]), np.concatenate([b, weight * t_obs])
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    t = sol[2 * K:]
    return t - P_GAUGE @ t


K_LM, LOC_ERR = 4, 1.5                                               # 4 landmarks, 1.5 nm localisation error
XZ = rng8.uniform(-25, 25, (K_LM, 2))
u_tracks = XZ[:, :1] * np.cos(beta)[None] + XZ[:, 1:] * np.sin(beta)[None] + t_true[None]
u_tracks = u_tracks + rng8.normal(0, LOC_ERR, u_tracks.shape)
A_lm = landmark_design(K_LM, beta)
print(f"unknowns: {A_lm.shape[1]} ({2 * K_LM} landmark coordinates + {n_t} shifts); "
      f"rank of the system = {np.linalg.matrix_rank(A_lm)}  → {A_lm.shape[1] - np.linalg.matrix_rank(A_lm)} "
      f"undeterminable directions (the gauge)")
est_lm = fit_shifts(u_tracks, beta)
print(f"landmark-only shift error: {gauge_free_error(est_lm, t_true):.2f} nm")

🔍 **What just happened?** The linear system is missing exactly **two** ranks — the two gauge directions predicted
above. Landmarks pin down the *global geometry* robustly (no feedback loop through a reconstruction), but each track is
only as good as its localisation, and natural landmarks in a noisy cellular image are few and imprecise.

### 8.5 A joint toy: landmarks and intensities refining each other

Han *et al.* (2019) alternate the two kinds of evidence. Our simplified version: start from the landmark solution, then
repeat {one projection-matching round → fuse its shift estimates with the landmark equations in one weighted least-squares
fit}. The landmarks anchor the slowly varying error patterns; the intensities supply precise per-image detail.

In [ ]:
# ── 8.5 Joint refinement: landmark equations + projection-matching observations ──
est_joint, err_joint = est_lm.copy(), [gauge_free_error(est_lm, t_true)]
for it in range(6):
    t_obs = projection_matching_step(measured, est_joint, ang60)
    est_joint = fit_shifts(u_tracks, beta, t_obs=t_obs, weight=2.0)
    err_joint.append(gauge_free_error(est_joint, t_true))
rec_joint = reconstruct_with(est_joint)
print("joint: shift error (nm):", " → ".join(f"{e:.2f}" for e in err_joint))
summary = {"no alignment": (gauge_free_error(np.zeros(n_t), t_true), pearson(rec_misaligned, phantom)),
           "landmarks only": (err_joint[0], pearson(reconstruct_with(est_lm), phantom)),
           "projection matching only": (err_pm[-1], pearson(rec_pm, phantom)),
           "joint (landmarks + matching)": (err_joint[-1], pearson(rec_joint, phantom)),
           "perfect": (0.0, pearson(rec_perfect, phantom))}
print(f"\n{'method':30s} {'shift error (nm)':>17s} {'corr with truth':>16s}")
for k_, (e, cc) in summary.items():
    print(f"{k_:30s} {e:17.2f} {cc:16.3f}")

In [ ]:
# ── 8.6 Pictures: what alignment buys ──
fig, axes = plt.subplots(1, 5, figsize=(12.5, 2.9))
axes[0].plot(ang60, t_true, "k.", ms=4, label="true")
axes[0].plot(ang60, est_joint + P_GAUGE @ (t_true - est_joint), "-", color="tab:red", lw=1, label="joint estimate")
axes[0].set_xlabel("tilt (°)"); axes[0].set_ylabel("shift (nm)"); axes[0].legend(fontsize=7); axes[0].set_title("per-tilt shifts")
for ax, im, t in zip(axes[1:], [rec_misaligned, rec_pm, rec_joint, rec_perfect],
                     ["not aligned", "projection matching", "joint", "perfect shifts"]):
    ax.imshow(im); ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 2.8))
ax.plot(err_pm, "o-", label="projection matching (from zero)")
ax.plot(err_joint, "s-", label="joint (from landmarks)")
ax.axhline(err_joint[0], color="gray", ls=":", label="landmarks only")
ax.set_xlabel("round"); ax.set_ylabel("gauge-free shift error (nm)"); ax.set_yscale("log")
ax.set_yticks([0.3, 0.5, 1, 2, 5]); ax.set_yticklabels(["0.3", "0.5", "1", "2", "5"]); ax.legend(fontsize=7)
ax.set_title("alignment error per round")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without alignment the vesicle and the molecules are smeared beyond recognition. Projection
matching alone already recovers most of the image; landmarks alone are robust but limited by their localisation error;
the joint fit is the most accurate and stays stable from round to round. This complementarity — robust geometry from
landmarks, precision from intensities — is the heart of the paper below.

✏️ **Your turn.** Change `LOC_ERR` (landmark localisation error) to `0.5` or `3.0`, or `K_LM` to `2`, and re-run 8.4–8.6.
When do the landmarks help the joint method, and when do they hurt?

### 📄 Paper connection — Han *et al.* 2019: joint marker-free alignment

**Paper**: Han R, Bao Z, Zeng X, Niu T, Zhang F, Xu M, Gao X. *A joint method for marker-free alignment of tilt series in
electron tomography.* Bioinformatics 35(14):i249–i259 (ISMB 2019).
[doi:10.1093/bioinformatics/btz323](https://doi.org/10.1093/bioinformatics/btz323)

- **Question it answers:** how to align a tilt series **without gold fiducials**, when landmark positions and projection
  parameters depend on each other (earlier marker-free methods treated them separately).
- **What the paper actually does:** it **alternates** two steps until the projection model converges:
  1. **Track-based (extrinsic) alignment** — SIFT keypoints matched with a kd-tree (ratio 0.7), RANSAC-estimated
     view-to-view affine maps, landmarks sampled on the 0° view and propagated through the series; then **bundle
     adjustment** of the projection model above (pitch $\alpha$, tilt $\beta$, in-plane rotation $\gamma$, scale $s$,
     shift $\mathbf t$).
  2. **Intensity-based (intrinsic) refinement** — reconstruct small local volumes around landmarks, re-project them
     ("reprojection-invariant landmarks"; the paper proves that re-centring the 3D origin changes only the translation
     and keeps projections consistent), re-localise the landmarks and refine parameters by minimising
     $$\arg\min_{\{\alpha_j,\gamma_j,\mathbf t_j\}}\sum_j\sum_{u\in D}\Big|\mathcal T^{2d}_{\mathbf t_j}\big(\mathcal R^{2d}_{\gamma_j}(I_j)\big)(u)-\mathcal P^{3d}_{s_j,\alpha_j,\beta_j}(V)(u)\Big|^2,$$
     *in words:* rotate and shift each image $I_j$ so that, over a central region $D$, it best matches the projection of the
     reconstructed volume $V$.
- **What our toy keeps:** the projection model (reduced to one shift per image), bundle adjustment as linear least
  squares, projection matching against re-projections of a preliminary reconstruction, the gauge/re-centring freedom,
  and the idea that the two kinds of evidence refine each other.
- **What our toy leaves out:** real landmark detection and tracking (SIFT, RANSAC), the angles $\alpha,\beta,\gamma$ and
  scale, local reconstructions around landmarks, 2D images with two shift components, and the paper's actual alternation
  schedule. Our "fusion" by weighted least squares is our own simplification.
- **Result reported:** the authors report substantial gains in alignment accuracy and reconstruction quality over
  state-of-the-art methods on three real datasets (the introduction names IMOD and AuTom as the state of the art). We
  could not read the experiments section, so we quote no numbers.
- **Code:** <https://github.com/icthrm/joint-marker-free-alignment> (not part of AITom). Deep dive in this repo:
  [`../../TiltSeriesAlignment/`](../../TiltSeriesAlignment/).

---
## 9 · Annotate in 2D, get 3D: segmenting the tilt series instead of the tomogram

🧫 Suppose you want to find dense **calcium-phosphate granules** inside mitochondria. You could segment the
reconstructed tomogram — but it carries missing-wedge elongation and streaks. Alternatively, segment each **2D tilt
image** (which has no missing wedge, and where powerful 2D detectors and human annotation work well) and then combine the
2D masks into a 3D mask. A simple way to combine them: back-project every binary mask and keep voxels that fall inside
the masks of (almost) all views — the *visual hull* idea, also used for "shape from silhouettes" in computer vision.

In our toy both routes use simple thresholding, and both are allowed the same number of foreground voxels as the true
granules (an idealised, but fair, rule).

In [ ]:
# ── 9.1 A toy mitochondrion with dense granules, imaged at three noise levels ──
def dice(a: np.ndarray, b: np.ndarray) -> float:
    """Dice overlap 2|A∩B| / (|A|+|B|) of two boolean masks."""
    return float(2 * (a & b).sum() / (a.sum() + b.sum() + 1e-12))


S9 = 48
gz, gy, gx = np.meshgrid(*[np.arange(S9) - (S9 - 1) / 2] * 3, indexing="ij")
body = ((gz / 9) ** 2 + (gy / 18) ** 2 + (gx / 15) ** 2 <= 1).astype(np.float32)        # mitochondrion (nm)
granules = np.zeros_like(body)
for c in [(-3, -8, -6), (3, 6, -5), (0, 0, 7), (-2, 10, 6), (4, -9, 5)]:
    granules[((gz - c[0]) ** 2 + (gy - c[1]) ** 2 + (gx - c[2]) ** 2) <= 3.2 ** 2] = 1
mito = ndimage.gaussian_filter(0.25 * body + granules, 0.7)
truth9 = granules > 0.5
ang9 = np.arange(-60, 61, 3.0)
t0 = time.time()
tilts9 = project_tilt_series(mito, ang9)
true_2d = project_tilt_series(granules, ang9) > 0.5
results9, show9 = {}, {}
NOISE_LEVELS = (0.3, 1.0, 3.0)                                       # noise std / signal std, per tilt image
for noise in NOISE_LEVELS:
    tn = tilts9 + np.random.default_rng(9).normal(0, noise * tilts9.std(), tilts9.shape)
    # route A: reconstruct, smooth, threshold in 3D
    rec9 = ndimage.gaussian_filter(weighted_back_projection(tn, ang9, S9), 1.0)
    mask_3d = rec9 >= np.sort(rec9.ravel())[-truth9.sum()]
    # route B: segment every 2D tilt image (background-subtracted + Otsu), back-project the masks, keep the hull
    sm = ndimage.gaussian_filter(tn, (0, 1.0, 1.0))
    hp = sm - ndimage.gaussian_filter(sm, (0, 4, 4))
    masks2d = np.stack([im > threshold_otsu(im) for im in hp])
    frac = np.zeros((S9, S9, S9), np.float32)
    for m, a in zip(masks2d.astype(np.float32), ang9):
        frac += ndimage.rotate(np.broadcast_to(m[None], (S9, S9, S9)), -a, axes=(0, 2), reshape=False, order=1)
    frac /= len(ang9)
    mask_2d3d = frac >= np.sort(frac.ravel())[-truth9.sum()]
    results9[noise] = (dice(masks2d, true_2d), dice(mask_3d, truth9), dice(mask_2d3d, truth9))
    show9[noise] = (rec9, mask_3d, mask_2d3d)
print(f"done in {time.time() - t0:.1f} s")
print(f"{'noise / tilt image':>18s} {'2D masks Dice':>14s} {'3D route Dice':>14s} {'2D→3D route Dice':>17s}")
for noise, (d2, d3, d23) in results9.items():
    print(f"{noise:18.1f} {d2:14.3f} {d3:14.3f} {d23:17.3f}")

In [ ]:
# ── 9.2 Side views (xz) of the two routes ──
yc = int(S9 / 2 - 8 + 0.5)                                          # a slice through two granules
fig, axes = plt.subplots(2, 4, figsize=(10, 4.6))
for row, noise in enumerate((NOISE_LEVELS[0], NOISE_LEVELS[-1])):
    rec9, m3, m23 = show9[noise]
    d2, d3, d23 = results9[noise]
    panels = [(mito[:, yc], "truth (xz slice)"), (rec9[:, yc], f"tomogram, noise {noise}"),
              (m3[:, yc], f"3D route (Dice {d3:.2f})"), (m23[:, yc], f"2D→3D route (Dice {d23:.2f})")]
    for col, (ax, (im, t)) in enumerate(zip(axes[row], panels)):
        ax.imshow(im); ax.set_title(t, fontsize=8); ax.set_xticks([]); ax.set_yticks([])
        if col >= 2:                                                # outline the true granules on the masks
            ax.contour(truth9[:, yc], levels=[0.5], colors="tab:orange", linewidths=0.8)
    axes[row, 0].set_ylabel("z (nm)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** (Orange contours = true granules.) When each 2D tilt image can be segmented accurately (low
noise), the 2D→3D route beats thresholding the tomogram, because its masks avoid the reconstruction's streaks and
elongation. Even when individual 2D masks are mediocre (middle noise level), requiring agreement across all 41 views is
itself a kind of averaging and the 2D→3D route still holds up. When the tilt images become too noisy for our simple 2D
thresholding (highest level), the 3D route wins: reconstruction *averages* all views *before* deciding. The lesson is not "2D is better" or "3D is better" — it is that **where you annotate** (images vs
tomogram) is a design choice with trade-offs, and that a strong 2D detector shifts the balance toward 2D.

### 📄 Paper connection — Zeng *et al.* 2022: structure detection from 2D annotated tilt series

**Paper**: Zeng X, Lin Z, Uddin MR, Zhou B, Cheng C, Zhang J, Freyberg Z, Xu M. *Structure detection in three-dimensional
cellular cryoelectron tomograms by reconstructing two-dimensional annotated tilt series.* Journal of Computational Biology
29(8):932–941 (2022). [doi:10.1089/cmb.2021.0606](https://doi.org/10.1089/cmb.2021.0606)

> ⚠️ We could not access the full text of this paper while writing; the description below is limited to the abstract.
> Read the paper before relying on details.

- **Question it answers:** detection directly in reconstructed 3D tomograms suffers from reconstruction artifacts and
  information loss, and most strong detectors are 2D — can we detect in 2D and lift the result to 3D?
- **What the paper does (abstract level):** a "2D-to-3D" framework — run 2D models on each tilt image, then map the 2D
  annotations back into 3D by reconstructing the annotated tilt series; instantiated as semantic segmentation (of
  mitochondrial calcium-phosphate granules), edge detection (spherical edges) and object localisation (mitochondria).
  The exact reconstruction step used for the annotations is not described in the abstract.
- **What our toy keeps:** annotate tilt images, combine masks in 3D, compare with annotating the tomogram.
- **What our toy leaves out:** learned 2D models, the paper's reconstruction of annotations (our visual hull is one
  plausible choice, not necessarily theirs), real data.
- **Result reported:** the authors report that segmentation on the 2D projections outperformed the approach on 3D
  tomograms, and "promising" results for localisation and edge detection (no numbers verified).
- **Code:** none found.

---
## ✅ Checkpoint

1. A 200 nm lamella is tilted to 60°. How thick is the ice the beam crosses?
   <details><summary>Answer</summary>200 / cos 60° = 400 nm — twice the 0° thickness.</details>
2. State the projection-slice theorem in one sentence, and say what a tilt series covers in 3D Fourier space.
   <details><summary>Answer</summary>The Fourier transform of a projection equals the central slice of the object's Fourier transform perpendicular to the projection direction. A single-axis tilt series covers a fan of central planes that all contain the tilt axis (k_y), leaving a missing wedge around k_z.</details>
3. Why does weighted back-projection multiply each projection's spectrum by |k|?
   <details><summary>Answer</summary>The measured central lines/planes are densely packed near the origin and sparse at high frequency (density ∝ 1/|k|); the ramp |k| compensates for this over-representation of low frequencies.</details>
4. In a tomogram with tilts ±60° about y, which parts of a spherical vesicle disappear, and why?
   <details><summary>Answer</summary>The top and bottom (roughly horizontal membrane patches): their Fourier content lies along k_z, inside the missing wedge. The sides, whose content lies along k_x, survive.</details>
5. You must split 120 e⁻/Å² over a tilt series. Does using 61 instead of 31 images make the tomogram noisier?
   <details><summary>Answer</summary>Not to first order: by dose fractionation the noise of a linear reconstruction depends on the total dose, not on how it is split. More images improve angular sampling (Crowther), at the cost of more dimmer images that are harder to align.</details>
6. Why can tilt-series alignment never determine all shifts uniquely?
   <details><summary>Answer</summary>Translating the whole 3D object by (a, b) in (x, z) shifts image j by a·cosβ_j + b·sinβ_j; such shift patterns are indistinguishable from object motion (the gauge freedom), so they only re-centre the tomogram.</details>

## 🧠 Questions & discussion

1. The missing wedge removes information *along the beam*. Which biological structures (membranes, filaments, flat
   assemblies) are most at risk of being misinterpreted, and how would you design a tilt experiment to check?
2. SIRT with a positivity constraint looked better than WBP in our toy. Why might positivity be a *poor* assumption for
   real cryo-ET data (hint: CTF, mean subtraction, ice)?
3. Gold beads make alignment easy but create strong streak artifacts. When would you choose a marker-free method?
4. In Section 8, projection matching uses the reconstruction to align the images that make the reconstruction. Where
   else in the course might such a "self-consistency" loop create bias? (Think of Module 14's reference bias.)
5. Zeng *et al.* segment the 2D tilt images. List one advantage and one disadvantage of 2D annotation compared with 3D.

## 🚀 Bonus — going further

- **Deep dives in this repo:** [`../../MissingWedgeReconstruction/`](../../MissingWedgeReconstruction/) (WBP and SIRT in more
  depth), [`../../TiltSeriesAlignment/`](../../TiltSeriesAlignment/), [`../../TomogramDiagnosisBuild/`](../../TomogramDiagnosisBuild/)
  (a practical build-and-diagnose workflow) and [`../../3DCryoEMReconstruction/`](../../3DCryoEMReconstruction/) (the
  single-particle counterpart: Fourier slice insertion).
- **Add in-plane rotation:** extend the alignment toy to 2D tilt images with shifts in x and y and a small in-plane
  rotation $\gamma_j$ per image; estimate $\gamma_j$ by a 1D search before the shift.
- **Dual-axis tomography:** reconstruct the 3D specimen from two orthogonal ±60° series (rotate the volume 90° about z for
  the second) and compare the xz slices with Section 5.2.
- **Real code:** run the official joint marker-free alignment
  (<https://github.com/icthrm/joint-marker-free-alignment>) or IMOD's `etomo` on a public tilt series from
  [EMPIAR](https://www.ebi.ac.uk/empiar/) or the [CZ cryoET Data Portal](https://cryoetdataportal.czscience.com/).
- **Noise in WBP:** compute the radial noise power spectrum of a WBP reconstruction of pure noise. Why does it rise with
  frequency? (Module 09 compares this "coloured" noise with the white noise of the fast simulator.)

## 📝 Lab note

```text
Module 07 — date:
Input: 2D slice specimen (zoo molecules + vesicle); 64³ specimen of 4 zoo molecules; misaligned tilt series
Output: sinograms, BP/WBP/SIRT reconstructions, missing-wedge spectra, PSF elongation, aligned shifts, 2D→3D masks
Method: projection-slice theorem, ramp-filtered back-projection, SIRT, projection matching + landmark least squares
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Han et al. 2019 (joint marker-free alignment); Zeng et al. 2022 (2D annotated tilt series)
```

## 📚 References

1. Han R, Bao Z, Zeng X, Niu T, Zhang F, Xu M, Gao X. A joint method for marker-free alignment of tilt series in electron tomography. *Bioinformatics* 35(14):i249–i259 (2019). doi:10.1093/bioinformatics/btz323
2. Zeng X, Lin Z, Uddin MR, Zhou B, Cheng C, Zhang J, Freyberg Z, Xu M. Structure detection in three-dimensional cellular cryoelectron tomograms by reconstructing two-dimensional annotated tilt series. *J Comput Biol* 29(8):932–941 (2022). doi:10.1089/cmb.2021.0606
3. Zhan X, Jiang R, Gupta V, Swaminathan T, Wang Y, Zhang G, Wang H, Xu M. MicroFM: Physics-guided flow matching for isotropic microscopy reconstruction. *CVPR* (2026).
4. Crowther RA, DeRosier DJ, Klug A. The reconstruction of a three-dimensional structure from projections and its application to electron microscopy. *Proc R Soc Lond A* 317:319–340 (1970). doi:10.1098/rspa.1970.0119
5. Radermacher M. Three-dimensional reconstruction of single particles from random and nonrandom tilt series. *J Electron Microsc Tech* 9:359–394 (1988). doi:10.1002/jemt.1060090405
6. Hegerl R, Hoppe W. Influence of electron noise on three-dimensional image reconstruction. *Z Naturforsch A* 31:1717–1721 (1976).
7. Gilbert P. Iterative methods for the three-dimensional reconstruction of an object from projections. *J Theor Biol* 36:105–117 (1972). doi:10.1016/0022-5193(72)90180-4
8. Hagen WJH, Wan W, Briggs JAG. Implementation of a cryo-electron tomography tilt-scheme optimized for high resolution subtomogram averaging. *J Struct Biol* 197:191–198 (2017). doi:10.1016/j.jsb.2016.06.007
9. Mastronarde DN, Held SR. Automated tilt series alignment and tomographic reconstruction in IMOD. *J Struct Biol* 197:102–113 (2017). doi:10.1016/j.jsb.2016.07.011
10. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast volumetric matching. *J Struct Biol* 178:152–164 (2012) — source of the missing-wedge mask formula used in the toolkit (Module 12).

## ➡️ Next

**[08 · Scanning the beam: 4D-STEM, shadow images and depth sectioning](08_4dstem_shadow_tomography.ipynb)** — a
different way to image the same specimens: scan a focused probe, record a whole diffraction pattern at every position,
and recover depth from a single scan.
After that, [09 · Build an in silico cell](09_in_silico_cell_simulation.ipynb) packs hundreds of molecules into a
cell-sized box and pushes them through the full physics of this module and Module 06.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")